# Observation series and intelligence r-GCN classification network with LLM explanations

This notebook is the relation-aware variant of `observation_series_and_intel_rgcn_classification_advanced_network.ipynb`. It replaces homogeneous GraphSAGE message passing with **relational graph convolution (`RGCNConv`)**, so observation, intelligence, platform, emitter, and temporal edge types learn separate transformations.

The classifier is evidential: non-negative evidence produces Dirichlet parameters. The explanation layer reports the candidate emitter, belief masses, Dempster–Shafer (DS) ignorance, probability margin, supporting/contradicting observations, and collection advice. An LLM may verbalize the supplied facts, but it is explicitly prohibited from changing scores or inventing evidence.


## Environment

Install the PyTorch build appropriate for the host first, then install PyTorch Geometric. The remaining cells assume `torch` and `torch_geometric` are available.


In [ ]:
# Uncomment in a compatible notebook runtime.
# %pip install torch torch-geometric pandas numpy


In [ ]:
from __future__ import annotations

import json
import math
from dataclasses import asdict, dataclass
from typing import Any, Mapping, Sequence

import torch
from torch import Tensor, nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import RGCNConv


## Relation-aware graph contract

`edge_type[i]` is the integer relation for `edge_index[:, i]`. Keep forward and reverse relations distinct: direction carries operational meaning. Production preprocessing should create the same mapping for training and inference and persist it beside the checkpoint.


In [ ]:
RELATIONS = {
    "observation_detects_emitter": 0,
    "emitter_detected_by_observation": 1,
    "intel_supports_emitter": 2,
    "emitter_supported_by_intel": 3,
    "emitter_installed_on_platform": 4,
    "platform_carries_emitter": 5,
    "observation_precedes_observation": 6,
    "observation_follows_observation": 7,
}
NUM_RELATIONS = len(RELATIONS)

def validate_graph(data: Data, num_relations: int = NUM_RELATIONS) -> None:
    if data.edge_type.dtype != torch.long:
        raise TypeError("edge_type must be torch.long")
    if data.edge_index.shape[1] != data.edge_type.numel():
        raise ValueError("Every edge must have exactly one relation type")
    if data.edge_type.numel() and not (0 <= int(data.edge_type.min()) <= int(data.edge_type.max()) < num_relations):
        raise ValueError("edge_type contains a relation outside the configured vocabulary")


## r-GCN evidential classifier

Unlike `SAGEConv`, each `RGCNConv` consumes `edge_type` and applies relation-specific weights. Basis decomposition limits parameters when the relation vocabulary grows. `softplus` converts logits into evidence; `alpha = evidence + 1` defines a Dirichlet distribution.


In [ ]:
class EvidentialRGCN(nn.Module):
    def __init__(
        self,
        in_channels: int,
        hidden_channels: int,
        num_classes: int,
        num_relations: int,
        num_bases: int | None = None,
        dropout: float = 0.2,
    ) -> None:
        super().__init__()
        bases = num_bases or min(8, num_relations)
        self.conv1 = RGCNConv(in_channels, hidden_channels, num_relations, num_bases=bases)
        self.conv2 = RGCNConv(hidden_channels, hidden_channels, num_relations, num_bases=bases)
        self.classifier = nn.Linear(hidden_channels, num_classes)
        self.dropout = dropout

    def forward(self, x: Tensor, edge_index: Tensor, edge_type: Tensor) -> dict[str, Tensor]:
        h = self.conv1(x, edge_index, edge_type)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = self.conv2(h, edge_index, edge_type)
        h = F.relu(h)
        evidence = F.softplus(self.classifier(h))
        alpha = evidence + 1.0
        strength = alpha.sum(dim=-1, keepdim=True)
        probabilities = alpha / strength
        ignorance = alpha.shape[-1] / strength
        belief = evidence / strength
        return {
            "evidence": evidence,
            "alpha": alpha,
            "probabilities": probabilities,
            "belief": belief,
            "ignorance": ignorance.squeeze(-1),
        }


## Evidential objective

Expected cross entropy trains the class assignment while KL annealing discourages unjustified evidence on incorrect classes. Validation and calibration should determine the annealing schedule and operational thresholds.


In [ ]:
def dirichlet_kl_to_uniform(alpha: Tensor) -> Tensor:
    classes = alpha.shape[-1]
    uniform = torch.ones_like(alpha)
    sum_alpha = alpha.sum(dim=-1, keepdim=True)
    sum_uniform = uniform.sum(dim=-1, keepdim=True)
    log_beta_alpha = torch.lgamma(sum_alpha) - torch.lgamma(alpha).sum(dim=-1, keepdim=True)
    log_beta_uniform = torch.lgamma(sum_uniform) - torch.lgamma(uniform).sum(dim=-1, keepdim=True)
    digamma_term = ((alpha - uniform) * (torch.digamma(alpha) - torch.digamma(sum_alpha))).sum(dim=-1, keepdim=True)
    return (log_beta_alpha - log_beta_uniform + digamma_term).squeeze(-1)

def evidential_loss(alpha: Tensor, target: Tensor, annealing: float = 0.1) -> Tensor:
    one_hot = F.one_hot(target, num_classes=alpha.shape[-1]).to(alpha.dtype)
    strength = alpha.sum(dim=-1, keepdim=True)
    expected_ce = (one_hot * (torch.digamma(strength) - torch.digamma(alpha))).sum(dim=-1)
    non_target_alpha = one_hot + (1.0 - one_hot) * alpha
    return (expected_ce + annealing * dirichlet_kl_to_uniform(non_target_alpha)).mean()


## Dirichlet and Dempster–Shafer decision summary

For $K$ singleton emitter hypotheses, DS belief is $b_k=e_k/S$ and uncommitted mass (ignorance) is $u=K/S$, where $S=\sum_k(e_k+1)$. Thus $\sum b_k+u=1$. Predictive probability is $p_k=\alpha_k/S=b_k+u/K$. High probability alone is not presented as certainty: the explanation also exposes ignorance, evidence strength, entropy, and the lead over the runner-up.


In [ ]:
@dataclass(frozen=True)
class EvidentialFinding:
    emitter: str
    probability: float
    belief_mass: float
    ignorance_mass: float
    dirichlet_strength: float
    normalized_entropy: float
    runner_up: str | None
    probability_margin: float
    confidence_band: str
    extreme_uncertainty: bool


def summarize_dirichlet(alpha: Tensor | Sequence[float], labels: Sequence[str]) -> EvidentialFinding:
    values = torch.as_tensor(alpha, dtype=torch.float64).flatten()
    if len(labels) != values.numel() or values.numel() < 2:
        raise ValueError("labels must match at least two Dirichlet parameters")
    if not torch.isfinite(values).all() or (values <= 0).any():
        raise ValueError("Dirichlet parameters must be finite and positive")
    strength = float(values.sum())
    classes = values.numel()
    probabilities = values / strength
    evidence = torch.clamp(values - 1.0, min=0.0)
    beliefs = evidence / strength
    ignorance = min(1.0, classes / strength)
    order = torch.argsort(probabilities, descending=True)
    top, second = int(order[0]), int(order[1])
    entropy = float(-(probabilities * probabilities.clamp_min(1e-12).log()).sum() / math.log(classes))
    margin = float(probabilities[top] - probabilities[second])
    extreme = ignorance >= 0.60 or entropy >= 0.90 or margin <= 0.05
    if extreme:
        band = "extreme uncertainty"
    elif ignorance >= 0.35 or entropy >= 0.75 or margin <= 0.15:
        band = "low confidence"
    elif ignorance >= 0.15 or margin <= 0.30:
        band = "moderate confidence"
    else:
        band = "high confidence"
    return EvidentialFinding(
        emitter=str(labels[top]), probability=float(probabilities[top]), belief_mass=float(beliefs[top]),
        ignorance_mass=ignorance, dirichlet_strength=strength, normalized_entropy=entropy,
        runner_up=str(labels[second]), probability_margin=margin, confidence_band=band,
        extreme_uncertainty=extreme,
    )


## Evidence-grounded LLM explanation layer

The deterministic payload is the safety boundary. It gives the LLM only model outputs and retrieved evidence, demands calibrated language, and requires active collection advice under extreme uncertainty. The LLM is an explanation layer—not an estimator or decision authority. `llm_client` is injected so this notebook does not bind the pipeline to a vendor; it must provide `complete(prompt) -> str`.

Collection recommendations are conditional on authorization and operational safety. Active RADAR can reveal position or intent, so a human operator must decide whether its information gain outweighs that risk.


In [ ]:
def build_explanation_payload(
    alpha: Tensor | Sequence[float],
    labels: Sequence[str],
    supporting_evidence: Sequence[str] = (),
    contradicting_evidence: Sequence[str] = (),
    missing_evidence: Sequence[str] = (),
) -> dict[str, Any]:
    finding = summarize_dirichlet(alpha, labels)
    actions = list(missing_evidence)
    if finding.extreme_uncertainty:
        actions.extend([
            "If authorized and tactically safe, engage or retask an active RADAR for range, track, and signature evidence.",
            "Collect another time-separated passive RF observation and improve angle-of-arrival geometry.",
            "Cross-cue independent EO/IR, IFF, or intelligence sources before acting on identity.",
        ])
    actions = list(dict.fromkeys(actions))
    record = {
        "finding": asdict(finding),
        "supporting_evidence": list(supporting_evidence),
        "contradicting_evidence": list(contradicting_evidence),
        "recommended_collection": actions,
        "decision_warning": "Identification is probabilistic; retain human review and do not treat the LLM narrative as new evidence.",
    }
    record["llm_prompt"] = f"""You are explaining an evidential r-GCN emitter-classification result.
Use only the JSON facts below. Do not alter numbers, infer absent facts, or claim certainty.
Start with 'Most likely emitter: <name>' and say this is a hypothesis.
Explain probability, DS belief mass, DS ignorance, Dirichlet strength, entropy, and runner-up margin in plain language.
Distinguish supporting, contradicting, and missing evidence. State when a list is empty.
Calibrate wording to confidence_band. If extreme_uncertainty is true, explicitly say identification is unreliable and recommend the listed collection actions, including active RADAR only with its authorization/safety caveat.
End with the decision warning. Never present your prose as additional evidence.
FACTS:
{json.dumps(record, indent=2, sort_keys=True)}"""
    return record


def deterministic_explanation(payload: Mapping[str, Any]) -> str:
    f = payload["finding"]
    support = "; ".join(payload["supporting_evidence"]) or "none supplied"
    contradict = "; ".join(payload["contradicting_evidence"]) or "none supplied"
    text = (
        f"Most likely emitter: {f['emitter']} (hypothesis), probability {f['probability']:.1%}. "
        f"Confidence is {f['confidence_band']}: DS committed belief is {f['belief_mass']:.1%}, "
        f"uncommitted/ignorance mass is {f['ignorance_mass']:.1%}, Dirichlet strength is "
        f"{f['dirichlet_strength']:.2f}, normalized entropy is {f['normalized_entropy']:.2f}, and "
        f"the lead over {f['runner_up']} is {f['probability_margin']:.1%}. "
        f"Supporting evidence: {support}. Contradicting evidence: {contradict}."
    )
    if f["extreme_uncertainty"]:
        text += " Identification is unreliable at this uncertainty level. Recommended collection: " + " ".join(payload["recommended_collection"])
    return text + " " + payload["decision_warning"]


def explain_with_llm(payload: Mapping[str, Any], llm_client: Any | None = None) -> str:
    if llm_client is None:
        return deterministic_explanation(payload)
    return str(llm_client.complete(payload["llm_prompt"]))


## Example: extreme uncertainty

Equal, low-strength Dirichlet parameters intentionally demonstrate the guardrail. This produces high ignorance and prompts additional collection, including a conditional active-RADAR recommendation.


In [ ]:
emitter_labels = ["N019 Slot Back", "AN/APG-68", "RDM"]
uncertain_alpha = torch.tensor([1.20, 1.15, 1.10])
payload = build_explanation_payload(
    uncertain_alpha,
    emitter_labels,
    supporting_evidence=["One intermittent X-band pulse train was observed."],
    contradicting_evidence=["Pulse repetition interval overlaps multiple candidate families."],
    missing_evidence=["No independent range, IFF, or EO/IR correlation is available."],
)
print(deterministic_explanation(payload))


## Connecting inference to explanations

Select only emitter nodes (or graph-level pooled outputs) before explanation. Preserve the entire `alpha` vector rather than passing only an argmax: uncertainty cannot be reconstructed from a winning label. Store the relation map, class labels, checkpoint ID, calibration version, evidence provenance, and threshold configuration with every result.


In [ ]:
def explain_emitter_node(
    model: EvidentialRGCN,
    graph: Data,
    node_index: int,
    emitter_labels: Sequence[str],
    supporting_evidence: Sequence[str] = (),
    contradicting_evidence: Sequence[str] = (),
    missing_evidence: Sequence[str] = (),
    llm_client: Any | None = None,
) -> tuple[dict[str, Any], str]:
    validate_graph(graph)
    model.eval()
    with torch.no_grad():
        output = model(graph.x, graph.edge_index, graph.edge_type)
    payload = build_explanation_payload(
        output["alpha"][node_index].cpu(), emitter_labels, supporting_evidence,
        contradicting_evidence, missing_evidence,
    )
    return payload, explain_with_llm(payload, llm_client)


## Operational validation checklist

1. Split evaluation by scenario/time to prevent observation-series leakage.
2. Report accuracy and macro-F1 alongside NLL, Brier score, expected calibration error, and selective-risk curves.
3. Test relation ablations and verify that shuffled `edge_type` reduces performance.
4. Tune uncertainty/action thresholds on held-out operationally representative data; the example thresholds are not doctrine.
5. Exercise known, ambiguous, conflicting, out-of-distribution, sensor-failure, and adversarial cases.
6. Audit every narrative against its payload; the LLM must never add evidence or overstate identity.
7. Require human approval for consequential decisions and for active-sensor collection.
